# Tarea 2 — Procesamiento con spaCy

Se carga `es_core_news_lg` y se procesa el corpus con `nlp.pipe()`. Para la pregunta del trabajo se utilizan tres operaciones: POS, lematización y morfología.

La tabla principal compara la frecuencia absoluta y relativa de adjetivos (ADJ) y adverbios (ADV) por obra.

In [ ]:
import json
from collections import Counter

import pandas as pd
import spacy


In [ ]:
CORPUS_PATH = "corpus.jsonl"
MODEL = "es_core_news_lg"

corpus = pd.read_json(CORPUS_PATH, lines=True)

required = {"id", "titulo", "autor", "texto"}
missing = required - set(corpus.columns)
if missing:
    raise ValueError(f"Faltan columnas requeridas en corpus.jsonl: {sorted(missing)}")

corpus[["id", "titulo", "autor"]].head()

In [ ]:
nlp = spacy.load(MODEL)
print(nlp.meta["name"], nlp.meta["version"])


## 1. POS: adjetivos y adverbios

Se procesan los documentos de forma secuencial con `nlp.pipe()`. Se cuentan los tokens etiquetados como `ADJ` y `ADV` y se calculan sus frecuencias relativas sobre el total de tokens lingüísticos procesados.

In [ ]:
pos_rows = []

texts = corpus["texto"].fillna("").tolist()
rows = corpus[["id", "titulo", "autor"]].to_dict("records")

for row, doc in zip(rows, nlp.pipe(texts, batch_size=16)):
    total = len(doc)
    counts = Counter(token.pos_ for token in doc)
    pos_rows.append({
        "id": row["id"],
        "titulo": row["titulo"],
        "autor": row["autor"],
        "tokens": total,
        "ADJ": counts["ADJ"],
        "ADV": counts["ADV"],
        "ADJ_rel": counts["ADJ"] / total if total else 0,
        "ADV_rel": counts["ADV"] / total if total else 0,
    })

tabla_pos = pd.DataFrame(pos_rows)
tabla_pos

## 2. Lemas

La lematización permite agrupar distintas formas flexionadas bajo una misma forma base. Para mantener la tabla enfocada en la pregunta, se muestran los lemas más frecuentes de ADJ y ADV por obra.

In [ ]:
lemma_counts = Counter()

for row, doc in zip(rows, nlp.pipe(texts, batch_size=16)):
    for token in doc:
        if token.pos_ in {"ADJ", "ADV"} and not token.is_punct and not token.is_space:
            lemma_counts[(row["titulo"], token.pos_, token.lemma_.lower())] += 1

tabla_lemmas = (
    pd.DataFrame(
        [
            {"titulo": titulo, "pos": pos, "lemma": lemma, "frecuencia": freq}
            for (titulo, pos, lemma), freq in lemma_counts.items()
        ]
    )
    .sort_values(["titulo", "pos", "frecuencia"], ascending=[True, True, False])
    .groupby(["titulo", "pos"], group_keys=False)
    .head(20)
    .reset_index(drop=True)
]
tabla_lemmas

## 3. Morfología

Se extraen rasgos morfológicos disponibles para ADJ y ADV. La tabla permite observar, por ejemplo, la distribución de género y número en los adjetivos.

In [ ]:
morph_counts = Counter()

for row, doc in zip(rows, nlp.pipe(texts, batch_size=16)):
    for token in doc:
        if token.pos_ not in {"ADJ", "ADV"} or token.is_punct or token.is_space:
            continue

        morph = token.morph.to_dict()
        genero = morph.get("Gender", "—")
        numero = morph.get("Number", "—")
        grado = morph.get("Degree", "—")
        morph_counts[(row["titulo"], token.pos_, genero, numero, grado)] += 1

tabla_morfologia = pd.DataFrame(
    [
        {
            "titulo": titulo,
            "pos": pos,
            "genero": genero,
            "numero": numero,
            "grado": grado,
            "frecuencia": frecuencia,
        }
        for (titulo, pos, genero, numero, grado), frecuencia in morph_counts.items()
    ]
).sort_values(["titulo", "pos", "frecuencia"], ascending=[True, True, False]).reset_index(drop=True)
tabla_morfologia

## Evidencia de procesamiento

Las tres variables siguientes son las tablas pandas que se pueden presentar como evidencia del procesamiento con spaCy:

- `tabla_pos`: comparación de ADJ y ADV por obra.
- `tabla_lemmas`: lemas más frecuentes de ADJ y ADV.
- `tabla_morfologia`: rasgos morfológicos de ADJ y ADV.

In [ ]:
tabla_pos.head(12), tabla_lemmas.head(20), tabla_morfologia.head(20)